In [4]:
"""
绘制单块电池 前/中/后期 IC 曲线对比图
展示:IC峰随老化下降(LAM) + 峰位横移(LLI)
用途:PPT / 论文插图,可复用
输出:{BID}_IC_evolution.png
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

# ===== 参数 =====
BID="025"           # 改成你想画的电池
DV=0.002; SIGMA=2; PROM=0.15; TOL=0.25
def TS(s): return pd.to_datetime(s, format='ISO8601')

def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=TS(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values
def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]
def is_3c(seg,cap_t,cap_v):
    t=TS(seg['Zeit'].iloc[0]).value
    It=3*np.interp(t,cap_t,cap_v); I=abs(seg['Strom'].mean())
    tt=(TS(seg['Zeit'])-TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60
def compute_ic(seg):
    """返回 电压网格, IC曲线, 峰位, 峰高"""
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(TS(seg['Zeit'])-TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    if len(Vu)<10: return None
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    if len(vgrid)<20: return None
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None
    peaks,_=find_peaks(IC,prominence=IC.max()*PROM)
    if len(peaks)==0: return None
    pk=peaks[np.argmax(IC[peaks])]
    return vgrid, IC, vgrid[pk], IC[pk]

# ===== 取前/中/后期各一个3C放电循环 =====
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
cap_t,cap_v=build_cap_interp(BID)
picks=[('Early life', 0, '#2166AC'),
       ('Mid life',   len(aging)//2, '#1B7837'),
       ('Late life',  len(aging)-1, '#B2182B')]

plt.rcParams.update({'font.size':12,'font.family':'Arial'})
fig,ax=plt.subplots(figsize=(8,5.5))
results=[]
for label,fi,color in picks:
    with fs.open(aging[fi]) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'])
    seg=None
    for s in get_raw(df,'DCH',-1):
        if is_3c(s,cap_t,cap_v): seg=s; break
    if seg is None: print(f"{label}: no 3C segment"); continue
    r=compute_ic(seg)
    if r is None: print(f"{label}: IC failed"); continue
    vgrid,IC,Vpeak,ICpeak=r
    results.append((label,Vpeak,ICpeak))
    ax.plot(vgrid,IC,color=color,lw=2.2,label=f'{label}  (peak {ICpeak:.2f})')
    ax.plot(Vpeak,ICpeak,'o',color=color,ms=9,markeredgecolor='white',markeredgewidth=1.5,zorder=5)
    print(f"{label}: Vpeak={Vpeak:.4f}V, peak height={ICpeak:.2f}")

ax.set_xlabel('Voltage (V)',fontsize=13)
ax.set_ylabel('Incremental Capacity  IC = |dQ/dV|',fontsize=13)
ax.set_title(f'IC Peak Evolution with Aging — Cell {BID}',fontsize=14,fontweight='bold',pad=12)
ax.legend(frameon=True,fontsize=11,loc='upper left')
ax.grid(alpha=0.25,linestyle='--')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 标注:峰高下降箭头
if len(results)>=2:
    txt="Peak height drops with aging\n(active material loss, LAM)"
    ax.annotate(txt, xy=(results[-1][1], results[-1][2]),
                xytext=(0.62,0.55), textcoords='axes fraction',
                fontsize=10.5, color='#555', ha='left',
                arrowprops=dict(arrowstyle='->',color='#888',lw=1.3))

plt.tight_layout()
plt.savefig(f'{BID}_IC_evolution.png',dpi=200,bbox_inches='tight')
print(f"\nSaved: {BID}_IC_evolution.png")

Early life: Vpeak=3.1951V, peak height=7.33
Mid life: Vpeak=3.1918V, peak height=6.73
Late life: Vpeak=3.1879V, peak height=6.04

Saved: 025_IC_evolution.png


In [5]:
"""
多工况电池 IC演化 网格对比图
每个电池一个子图,每个子图含早/中/晚期IC曲线
直观对比:不同工况的IC峰形态与演化差异
输出:IC_evolution_grid.png
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

DV=0.002; SIGMA=2; PROM=0.15; TOL=0.25
def TS(s): return pd.to_datetime(s, format='ISO8601')

# ===== 要对比的电池(改成你想展示的工况组合) =====
# 建议挑工况差异大的:深DOD / 浅DOD / 不同SOC / 不同温度
CELLS=['008','012','022','025']   # 你可以换/加,2-6个都行

def parse_cond(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return f"{m.group(1)}°C  {m.group(2)}%SOC  {m.group(3)}%DOD"
    return "?"
def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=TS(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values
def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]
def is_3c(seg,cap_t,cap_v):
    t=TS(seg['Zeit'].iloc[0]).value
    It=3*np.interp(t,cap_t,cap_v); I=abs(seg['Strom'].mean())
    tt=(TS(seg['Zeit'])-TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60
def compute_ic(seg):
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(TS(seg['Zeit'])-TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    if len(Vu)<10: return None
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    if len(vgrid)<20: return None
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None
    peaks,_=find_peaks(IC,prominence=IC.max()*PROM)
    if len(peaks)==0: return None
    pk=peaks[np.argmax(IC[peaks])]
    return vgrid, IC, vgrid[pk], IC[pk]

def get_ic_at(bid, fi, cap_t, cap_v):
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    fi=min(fi,len(aging)-1)
    with fs.open(aging[fi]) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'])
    for s in get_raw(df,'DCH',-1):
        if is_3c(s,cap_t,cap_v):
            return compute_ic(s)
    return None

# ===== 网格布局 =====
n=len(CELLS)
ncol=2 if n<=4 else 3
nrow=int(np.ceil(n/ncol))
plt.rcParams.update({'font.size':11,'font.family':'Arial'})
fig,axes=plt.subplots(nrow,ncol,figsize=(6.2*ncol,4.6*nrow))
axes=np.array(axes).reshape(-1)
stages=[('Early','#2166AC'),('Mid','#1B7837'),('Late','#B2182B')]

for idx,bid in enumerate(CELLS):
    ax=axes[idx]
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    cond=parse_cond(aging)
    cap_t,cap_v=build_cap_interp(bid)
    fis=[0, len(aging)//2, len(aging)-1]
    for (label,color),fi in zip(stages,fis):
        r=get_ic_at(bid,fi,cap_t,cap_v)
        if r is None: continue
        vgrid,IC,Vpeak,ICpeak=r
        ax.plot(vgrid,IC,color=color,lw=2,label=f'{label} ({ICpeak:.2f})')
        ax.plot(Vpeak,ICpeak,'o',color=color,ms=7,markeredgecolor='white',markeredgewidth=1.2,zorder=5)
    ax.set_title(f'Cell {bid}   {cond}',fontsize=12,fontweight='bold')
    ax.set_xlabel('Voltage (V)',fontsize=11)
    ax.set_ylabel('IC = |dQ/dV|',fontsize=11)
    ax.legend(fontsize=9,loc='upper left',title='Stage (peak)',title_fontsize=9)
    ax.grid(alpha=0.25,linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    print(f"Cell {bid} ({cond}) done")

# 关掉多余子图
for j in range(n,len(axes)): axes[j].axis('off')

fig.suptitle('IC Peak Evolution across Operating Conditions',
             fontsize=15,fontweight='bold',y=1.005)
plt.tight_layout()
plt.savefig('IC_evolution_grid.png',dpi=200,bbox_inches='tight')
print("\nSaved: IC_evolution_grid.png")

Cell 008 (35°C  70%SOC  60%DOD) done
Cell 012 (35°C  70%SOC  20%DOD) done
Cell 022 (35°C  30%SOC  60%DOD) done
Cell 025 (35°C  30%SOC  60%DOD) done

Saved: IC_evolution_grid.png


In [13]:
"""
回归原始数据：电流、电压、容量与裸 IC 的彻底解剖
强制统一横纵坐标轴，实现绝对公平的物理对比 (Apples-to-Apples)
输出: raw_data_debug.png
"""
import s3fs, numpy as np, pandas as pd
import pyarrow.parquet as pq
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import re
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

# S3 配置
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

def TS(s): return pd.to_datetime(s, format='ISO8601')

def parse_cond(bid):
    """自动解析电池的真实测试温度、SOC和DOD"""
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    if not aging: return "Unknown Condition"
    m = re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD', aging[0])
    if m:
        return f"{m.group(1)}°C {m.group(2)}%SOC {m.group(3)}%DOD"
    return "Unknown Condition"

def get_raw_dch(df):
    """提取大电流(3C)放电段"""
    m=((df['Zustand']=='DCH')&(df['Strom']<0)).values
    ch=np.diff(m.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    
    # 遍历所有放电段，必须是大于 100 个点，且平均电流大于 2.0A (过滤掉 0.5C 的 SOC 调整步)
    for s,e in zip(st,en):
        seg = df.iloc[s:e]
        if len(seg) > 100 and abs(seg['Strom'].mean()) > 2.0:
            return seg
    return None

def fetch_cycle_data(bid, fi):
    """读取指定文件的原始片段"""
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    if fi >= len(aging): fi = len(aging)-1
    
    with fs.open(aging[fi]) as fh:
        df = pq.ParquetFile(fh).read(columns=['Zeit','Spannung','Strom','Zustand']).to_pandas()
    
    seg = get_raw_dch(df)
    if seg is None: return None
    
    V = seg['Spannung'].values
    I = seg['Strom'].values
    tt = (TS(seg['Zeit']) - TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    
    # 1. 积分算容量 Q (Ah)
    Q = np.cumsum(np.abs(I) * np.diff(tt, prepend=tt[0]) / 3600.0)
    
    # 2. 网格化准备裸求导
    o = np.argsort(V); Vu, idx = np.unique(V[o], return_index=True); Qu = Q[o][idx]
    vgrid = np.arange(Vu.min(), Vu.max(), 0.002)
    Q_interp = np.interp(vgrid, Vu, Qu)
    
    # 3. 裸求导 (不加高斯平滑!)
    IC_raw = np.abs(np.gradient(Q_interp, vgrid))
    
    return tt, V, I, vgrid, Q_interp, IC_raw

# 选取你指定的两块电池
TEST_CELLS = ['020', '019']

# 布局：4行 (V, I, Q, IC) x 2列 (Cell)
plt.rcParams.update({'font.size':10, 'font.family':'sans-serif'})
fig, axes = plt.subplots(4, 2, figsize=(12, 14))
stages = [('Early', 0, '#2166AC'), ('Mid', 20, '#1B7837'), ('Late', -1, '#B2182B')]

# 用于记录全局最大值，强行统一坐标轴
global_max_t = 0
global_max_q = 0
global_max_ic = 0

for col, bid in enumerate(TEST_CELLS):
    # 动态获取真实的工况标签
    cond_label = parse_cond(bid)
    
    for label, fi, color in stages:
        data = fetch_cycle_data(bid, fi)
        if data is None: continue
        tt, V, I, vgrid, Q_interp, IC_raw = data
        
        # 记录最大值
        global_max_t = max(global_max_t, tt[-1])
        global_max_q = max(global_max_q, Q_interp.max())
        global_max_ic = max(global_max_ic, IC_raw.max())
        
        # Row 0: 电压 vs 时间
        axes[0, col].plot(tt, V, color=color, label=label)
        axes[0, col].set_ylabel('Raw Voltage (V)')
        axes[0, col].set_title(f"Cell {bid} ({cond_label})", fontweight='bold')
        
        # Row 1: 电流 vs 时间
        axes[1, col].plot(tt, I, color=color)
        axes[1, col].set_ylabel('Raw Current (A)')
        
        # Row 2: 积分容量 Q vs 电压 V
        axes[2, col].plot(vgrid, Q_interp, color=color)
        axes[2, col].set_ylabel('Capacity Q (Ah)')
        axes[2, col].set_xlabel('Voltage (V)')
        
        # Row 3: 裸 IC (dQ/dV) vs 电压 V
        axes[3, col].plot(vgrid, IC_raw, color=color, alpha=0.7)
        axes[3, col].set_ylabel('Raw IC = |dQ/dV| \n(No Gaussian)')
        axes[3, col].set_xlabel('Voltage (V)')

    axes[0, col].legend()
    for row in range(4):
        axes[row, col].grid(alpha=0.3, linestyle='--')

# ================= 核心修改：强制对齐坐标轴 =================
# 在所有图画完后，强行应用统一的比例尺
for col in range(2):
    # Row 0: 统一时间轴 和 电压区间
    axes[0, col].set_xlim(0, global_max_t * 1.05)
    axes[0, col].set_ylim(1.9, 3.4)
    
    # Row 1: 统一时间轴 和 电流区间 (假设 3C 是 3.3A 左右)
    axes[1, col].set_xlim(0, global_max_t * 1.05)
    axes[1, col].set_ylim(-4.0, -2.5)
    
    # Row 2: 统一电压横轴 (倒序) 和 容量纵轴
    axes[2, col].set_xlim(3.4, 1.9)
    axes[2, col].set_ylim(0, global_max_q * 1.1)
    
    # Row 3: 统一电压横轴 和 IC 纵轴
    axes[3, col].set_xlim(1.9, 3.4)
    axes[3, col].set_ylim(0, global_max_ic * 1.1)

plt.tight_layout()
plt.savefig('raw_data_debug.png', dpi=200, bbox_inches='tight')
print("Saved: raw_data_debug.png")

Saved: raw_data_debug.png


In [16]:
"""
聚焦相变核心区：IC峰值局部放大与演化追踪
自动定位 V_peak 并截取极窄电压窗口，完美展示 LAM (活性物质损失) 的几何坍塌
输出: ic_peak_zoom.png
"""
import s3fs, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import re
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

# S3 配置
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
fs=s3fs.S3FileSystem(**OPTS)

def TS(s): return pd.to_datetime(s, format='ISO8601')

def parse_cond(bid):
    """自动解析电池的真实测试温度，并计算实际的起止 SOC"""
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    if not aging: return "Unknown Condition"
    m = re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD', aging[0])
    if m:
        temp, avg_soc, dod = int(m.group(1)), int(m.group(2)), int(m.group(3))
        return f"{temp}°C | {dod}%DOD ({avg_soc+dod//2}% -> {avg_soc-dod//2}% SOC)"
    return "Unknown Condition"

def get_raw_dch(df):
    """提取大电流(3C)放电段"""
    m=((df['Zustand']=='DCH')&(df['Strom']<0)).values
    ch=np.diff(m.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    for s,e in zip(st,en):
        seg = df.iloc[s:e]
        if len(seg) > 100 and abs(seg['Strom'].mean()) > 2.0:
            return seg
    return None

def fetch_ic_smoothed(bid, fi):
    """读取并计算高斯平滑后的 IC 曲线，用于精准找峰"""
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    if fi >= len(aging): fi = len(aging)-1
    
    with fs.open(aging[fi]) as fh:
        df = pq.ParquetFile(fh).read(columns=['Zeit','Spannung','Strom','Zustand']).to_pandas()
    
    seg = get_raw_dch(df)
    if seg is None: return None
    
    V = seg['Spannung'].values
    I = seg['Strom'].values
    tt = (TS(seg['Zeit']) - TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    
    Q = np.cumsum(np.abs(I) * np.diff(tt, prepend=tt[0]) / 3600.0)
    o = np.argsort(V); Vu, idx = np.unique(V[o], return_index=True); Qu = Q[o][idx]
    vgrid = np.arange(Vu.min(), Vu.max(), 0.001) # 更密集的网格 1mV
    Q_interp = np.interp(vgrid, Vu, Qu)
    
    # 重新加入高斯平滑，只看宏观相变峰
    Q_smooth = gaussian_filter1d(Q_interp, sigma=3)
    IC_smooth = np.abs(np.gradient(Q_smooth, vgrid))
    
    return vgrid, Q_smooth, IC_smooth

# 选取你指定的两块电池 (10 和 20)
TEST_CELLS = ['008', '011']
stages = [('Early', 0, '#2166AC'), ('Mid', 20, '#1B7837'), ('Late', -1, '#B2182B')]

plt.rcParams.update({'font.size':11, 'font.family':'sans-serif'})
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, bid in enumerate(TEST_CELLS):
    cond_label = parse_cond(bid)
    ax = axes[col]
    
    v_peak_early = None # 用来锚定早期峰值的位置
    max_ic_display = 0  # 用来调整 Y 轴
    
    for label, fi, color in stages:
        data = fetch_ic_smoothed(bid, fi)
        if data is None: continue
        vgrid, Q_smooth, IC_smooth = data
        
        # 寻找平滑后的真实峰值
        peaks, properties = find_peaks(IC_smooth, prominence=0.5)
        if len(peaks) > 0:
            main_peak_idx = peaks[np.argmax(IC_smooth[peaks])]
            curr_v_peak = vgrid[main_peak_idx]
            curr_ic_peak = IC_smooth[main_peak_idx]
            
            # 记录第一个周期(Early)的峰位，作为局部放大的基准中心
            if label == 'Early':
                v_peak_early = curr_v_peak
            
            # 画线并打上峰值的圆点
            ax.plot(vgrid, IC_smooth, color=color, lw=2, label=f'{label} (Peak: {curr_v_peak:.3f}V)')
            ax.plot(curr_v_peak, curr_ic_peak, 'o', color=color, ms=6, markeredgecolor='white')
            
            max_ic_display = max(max_ic_display, curr_ic_peak)
        else:
            ax.plot(vgrid, IC_smooth, color=color, lw=2, linestyle='--', label=f'{label} (No Peak)')

    # ===== 核心：局部放大与对齐 (Dynamic Zoom-in) =====
    if v_peak_early is not None:
        # 以早期峰值为中心，左右各取 0.15V 的极窄窗口
        window_half_width = 0.15
        ax.set_xlim(v_peak_early - window_half_width, v_peak_early + window_half_width)
    else:
        # 如果连早期的峰都没找到（极其罕见），就给个默认区间
        ax.set_xlim(3.0, 3.4)
        
    ax.set_ylim(0, max_ic_display * 1.2)
    ax.set_title(f"Cell {bid}\n{cond_label}", fontweight='bold', pad=15)
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Smoothed IC = |dQ/dV|')
    ax.grid(alpha=0.3, linestyle='--')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('ic_peak_zoom.png', dpi=200, bbox_inches='tight')
print("Saved: ic_peak_zoom.png")

Saved: ic_peak_zoom.png


In [18]:
"""
两电池 × 四行物理量 对比图
行1: 电流-时间  行2: 电压-时间  行3: Q-电压  行4: IC-电压
左右两列横轴标尺统一(每行用相同xlim),便于对比
取每个电池 早/中/晚 三个老化阶段叠加
输出:two_cell_4row_compare.png
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

DV=0.002; SIGMA=2; PROM=0.15; TOL=0.25
def TS(s): return pd.to_datetime(s, format='ISO8601')

# ===== 两个要对比的电池 =====
CELL_L='014'    # 左列
CELL_R='012'    # 右列(挑工况差异大的,对比明显)

def parse_cond(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return f"{m.group(1)}°C  {m.group(2)}%SOC  {m.group(3)}%DOD"
    return "?"
def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=TS(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values
def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]
def is_3c(seg,cap_t,cap_v):
    t=TS(seg['Zeit'].iloc[0]).value
    It=3*np.interp(t,cap_t,cap_v); I=abs(seg['Strom'].mean())
    tt=(TS(seg['Zeit'])-TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60

def extract_curves(seg):
    """返回 t(s), V, I, Q(Ah), 以及IC的(vgrid,IC)"""
    t=(TS(seg['Zeit'])-TS(seg['Zeit'].iloc[0])).dt.total_seconds().values
    V=seg['Spannung'].values; I=seg['Strom'].values
    Q=np.cumsum(np.abs(I)*np.diff(t,prepend=t[0])/3600.0)
    # IC
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    ic=None
    if len(Vu)>=10:
        vgrid=np.arange(Vu.min(),Vu.max(),DV)
        if len(vgrid)>=20:
            Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
            IC=np.abs(np.gradient(Qsm,vgrid))
            ic=(vgrid,IC)
    return t,V,I,Q,ic

def get_stage_segments(bid):
    """取早/中/晚三个3C放电段"""
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    cond=parse_cond(aging); cap_t,cap_v=build_cap_interp(bid)
    fis=[0, len(aging)//2, len(aging)-1]
    out=[]
    for fi in fis:
        with fs.open(aging[fi]) as fh:
            df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                    columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
        df['Zeit']=pd.to_datetime(df['Zeit'])
        seg=None
        for s in get_raw(df,'DCH',-1):
            if is_3c(s,cap_t,cap_v): seg=s; break
        out.append(extract_curves(seg) if seg is not None else None)
    return cond,out

# ===== 取两电池数据 =====
condL,dataL=get_stage_segments(CELL_L)
condR,dataR=get_stage_segments(CELL_R)
print(f"L: Cell {CELL_L} {condL}")
print(f"R: Cell {CELL_R} {condR}")

stages=[('Early','#2166AC'),('Mid','#1B7837'),('Late','#B2182B')]

plt.rcParams.update({'font.size':10,'font.family':'Arial'})
fig,axes=plt.subplots(4,2,figsize=(11,13))

def plot_col(col, cond, data, bid):
    for (label,color),d in zip(stages,data):
        if d is None: continue
        t,V,I,Q,ic=d
        axes[0,col].plot(t, np.abs(I), color=color, lw=1.6, label=label)   # |电流|-时间
        axes[1,col].plot(t, V, color=color, lw=1.6, label=label)           # 电压-时间
        axes[2,col].plot(V, Q, color=color, lw=1.6, label=label)           # Q-电压
        if ic is not None:
            axes[3,col].plot(ic[0], ic[1], color=color, lw=1.6, label=label) # IC-电压
    axes[0,col].set_title(f'Cell {bid}   {cond}', fontsize=12, fontweight='bold')

plot_col(0, condL, dataL, CELL_L)
plot_col(1, condR, dataR, CELL_R)

# 轴标签
row_ylabel=['|Current| (A)','Voltage (V)','Q (Ah)','IC = |dQ/dV|']
row_xlabel=['Time (s)','Time (s)','Voltage (V)','Voltage (V)']
for r in range(4):
    for c in range(2):
        ax=axes[r,c]
        ax.grid(alpha=0.25,linestyle='--')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.set_xlabel(row_xlabel[r],fontsize=10)
        if c==0: ax.set_ylabel(row_ylabel[r],fontsize=11)
        ax.legend(fontsize=8,loc='best')

# ===== 关键:每行左右横轴标尺统一 =====
for r in range(4):
    x0=min(axes[r,0].get_xlim()[0], axes[r,1].get_xlim()[0])
    x1=max(axes[r,0].get_xlim()[1], axes[r,1].get_xlim()[1])
    axes[r,0].set_xlim(x0,x1); axes[r,1].set_xlim(x0,x1)
    # 行3、行4是电压横轴,也统一y轴更好对比(可选)
    # y也统一:
    y0=min(axes[r,0].get_ylim()[0], axes[r,1].get_ylim()[0])
    y1=max(axes[r,0].get_ylim()[1], axes[r,1].get_ylim()[1])
    axes[r,0].set_ylim(y0,y1); axes[r,1].set_ylim(y0,y1)

fig.suptitle(f'Discharge Signal Comparison — Cell {CELL_L} vs Cell {CELL_R}',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('two_cell_4row_compare.png',dpi=180,bbox_inches='tight')
print("\nSaved: two_cell_4row_compare.png")

L: Cell 014 35°C  70%SOC  20%DOD
R: Cell 012 35°C  70%SOC  20%DOD

Saved: two_cell_4row_compare.png


In [21]:
"""
单独提取 014 号电池的 放电时间(s) 与 电压(V) 原始数值
不仅在控制台打印采样点预览，还会将完整数据保存为 CSV 文件。
"""
import s3fs, numpy as np, pandas as pd
import pyarrow.parquet as pq
import warnings; warnings.filterwarnings('ignore')
import re
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

# S3 配置
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

BID = "014"
TOL = 0.25

def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=pd.to_datetime(c['CAP_start_time'], format='ISO8601'); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values

def get_raw_dch(df):
    """提取所有放电段"""
    m=((df['Zustand']=='DCH')&(df['Strom']<0)).values
    ch=np.diff(m.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]

def is_3c_dynamic(seg,cap_t,cap_v):
    """判断是否为动态3C大倍率放电"""
    t = seg['Zeit'].iloc[0].value # 获取纳秒级时间戳
    C_now = np.interp(t, cap_t, cap_v)
    It = 3 * C_now
    I = abs(seg['Strom'].mean())
    tt = (seg['Zeit'].iloc[-1] - seg['Zeit'].iloc[0]).total_seconds()
    return (It*(1-TOL) < I < It*(1+TOL)) and tt >= 60

def extract_timestamp_from_filename(filename):
    """
    【根据真实文件名修复】：从类似 "...=2024-12-20_013454=..." 的文件名中提取绝对时间戳。
    使用 YYYY-MM-DD_HHMMSS 格式进行字典序排序是物理上绝对安全的 chronological sorting。
    """
    match = re.search(r'(\d{4}-\d{2}-\d{2}_\d{6})', filename)
    if match:
        return match.group(1)
    return "" # 如果没找到时间戳，则返回空字符串（排在最前）

# ===== 主流程 =====
print(f"正在读取电池 {BID} 的循环数据...")
BASE = f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"

# 获取所有包含 Aging 和 Cyc 的文件
raw_files = [f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f]

# 使用提取到的真实物理时间戳对文件进行绝对时间排序
aging = sorted(raw_files, key=extract_timestamp_from_filename)

if not aging:
    print(f"未找到 {BID} 的老化文件！")
else:
    cap_t, cap_v = build_cap_interp(BID)
    
    # 提取排好序的第一个（最早期）和最后一个（最晚期）文件
    target_files = [aging[0], aging[-1]]
    labels = ["早期(Early)", "晚期(Late)"]

    for label, file_path in zip(labels, target_files):
        print(f"\n--- 处理 {label} 循环 ({file_path.split('=')[2]}) ---")
        with fs.open(file_path) as fh:
            df = next(pq.ParquetFile(fh).iter_batches(batch_size=500000, 
                    columns=['Zeit','Spannung','Strom','Zustand'])).to_pandas()
        
        # 统一将时间列预处理为 datetime 对象，避免后续重复计算
        df['Zeit'] = pd.to_datetime(df['Zeit'], format='ISO8601')
        
        # 找到这段数据里的 3C 放电段
        found = False
        for seg in get_raw_dch(df):
            if is_3c_dynamic(seg, cap_t, cap_v):
                # 提取相对时间(s)和电压(V)
                tt = (seg['Zeit'] - seg['Zeit'].iloc[0]).dt.total_seconds().values
                V = seg['Spannung'].values
                I = seg['Strom'].values
                
                # 打包成 DataFrame 方便打印和保存
                out_df = pd.DataFrame({'Time_s': tt, 'Voltage_V': V, 'Current_A': I})
                
                print(f"✅ 成功提取到放电段，共 {len(out_df)} 个数据点。")
                print("【开头5个数据点】:")
                print(out_df.head(5).to_string(index=False))
                print("...")
                print("【末尾5个数据点】:")
                print(out_df.tail(5).to_string(index=False))
                
                # 导出为 CSV
                csv_name = f"Cell_{BID}_{label}_VT_data.csv"
                out_df.to_csv(csv_name, index=False)
                print(f"💾 完整时间/电压数据已保存至本地: {csv_name}")
                
                found = True
                break # 只取该文件里的第一次有效放电
        
        if not found:
            print(f"❌ 在该文件中未找到符合 3C 标准的完整放电段。")

正在读取电池 014 的循环数据...

--- 处理 早期(Early) 循环 (2024-12-20_013454) ---
✅ 成功提取到放电段，共 551 个数据点。
【开头5个数据点】:
 Time_s  Voltage_V  Current_A
   0.00   3.208008  -3.554801
   0.68   3.199340  -3.556960
   0.83   3.197896  -3.557140
   1.02   3.196340  -3.557140
   1.71   3.191783  -3.557140
...
【末尾5个数据点】:
 Time_s  Voltage_V  Current_A
 239.64   2.516665   -3.55678
 239.69   2.515442   -3.55678
 239.99   2.505885   -3.55678
 240.01   2.503885   -3.55678
 240.04   2.503885   -3.55678
💾 完整时间/电压数据已保存至本地: Cell_014_早期(Early)_VT_data.csv

--- 处理 晚期(Late) 循环 (2026-02-15_152017) ---
✅ 成功提取到放电段，共 578 个数据点。
【开头5个数据点】:
 Time_s  Voltage_V  Current_A
   0.00   3.220788  -3.122215
   0.03   3.220788  -3.122215
   0.86   3.210564  -3.122934
   1.02   3.209453  -3.122934
   1.02   3.209453  -3.122934
...
【末尾5个数据点】:
 Time_s  Voltage_V  Current_A
 240.05   2.253618  -3.122574
 240.05   2.253618  -3.122574
 240.07   2.252285  -3.122574
 240.10   2.252285  -3.122574
 240.10   2.252285  -3.122574
💾 完整时间/电压数据已保存至本地: Cell